# 01 Automatic differentiation: forward and reverse mode

Lecture 10, training and generalization. Book chapters 2 (automatic
differentiation) and 3 (backpropagation in a multilayer perceptron).

**Goals**

- Compute the gradient of a small expression in forward mode (dual numbers)
  and in reverse mode (`Tensor.backward`), and compare the costs.
- Write the backward pass of a two-layer MLP by hand and check it against
  GlassNN's reverse mode and against finite differences (`gradcheck`).

**Expected finding** (asserted in `tests/test_examples.py::test_01_*`): the
hand-written backward pass, reverse-mode autodiff and central finite
differences agree.

In [ ]:
import numpy as np

import glassnn.functional as F
from glassnn import Tensor, backend
from glassnn.gradcheck import gradcheck

backend.set_default_dtype("float64")  # finite differences need float64
rng = np.random.default_rng(0)

## Forward mode with dual numbers

A dual number $a + \dot a\,\epsilon$ with $\epsilon^2 = 0$ carries a value and
a derivative (a *tangent*). Every operation propagates both, e.g.
$(a + \dot a \epsilon)(b + \dot b \epsilon) = ab + (\dot a b + a \dot b)\epsilon$.
Setting $\dot x_1 = 1, \dot x_2 = 0$ gives $\partial f / \partial x_1$; one
pass per input.

In [ ]:
class Dual:
    # A value and its tangent; just enough operations for f below.
    def __init__(self, value, tangent=0.0):
        self.value, self.tangent = value, tangent

    def __add__(self, other):
        return Dual(self.value + other.value, self.tangent + other.tangent)

    def __sub__(self, other):
        return Dual(self.value - other.value, self.tangent - other.tangent)

    def __mul__(self, other):
        return Dual(
            self.value * other.value,
            self.tangent * other.value + self.value * other.tangent,
        )

    def exp(self):
        return Dual(np.exp(self.value), np.exp(self.value) * self.tangent)

    def log(self):
        return Dual(np.log(self.value), self.tangent / self.value)


def f(x1, x2):
    # Works for Dual and for Tensor: both define *, +, -, exp and log.
    return x1.log() + x1 * x2 - x2.exp()


x1, x2 = 2.0, 0.5
forward = [
    float(f(Dual(x1, 1.0), Dual(x2, 0.0)).tangent),  # pass 1: d/dx1
    float(f(Dual(x1, 0.0), Dual(x2, 1.0)).tangent),  # pass 2: d/dx2
]
analytic = [1 / x1 + x2, float(x1 - np.exp(x2))]
print("forward mode:", forward)
print("analytic:    ", analytic)

## Reverse mode

`Tensor.backward()` records the graph during the forward pass and then
propagates *adjoints* $\bar z = \partial f / \partial z$ from the output to
the inputs: one backward pass gives the whole gradient.

In [ ]:
t1 = Tensor(x1, requires_grad=True)
t2 = Tensor(x2, requires_grad=True)
f(t1, t2).backward()
print("reverse mode:", [t1.grad.item(), t2.grad.item()])

For $f: \mathbb R^n \to \mathbb R$, forward mode needs $n$ passes and
reverse mode one: this is why neural networks, with millions of parameters
and one scalar loss, are trained with reverse mode (backpropagation). For
$f: \mathbb R \to \mathbb R^m$ the situation is the opposite.

## Backpropagation in a two-layer MLP, by hand

The network $\hat y = \tanh(X W_1^\top + b_1) W_2^\top + b_2$ with the mean
squared error. With $Z_1 = X W_1^\top + b_1$, $H = \tanh Z_1$, the backward
pass is (chapter 3):

$$
\bar{\hat y} = \tfrac{2}{N}(\hat y - y),\quad
\bar W_2 = \bar{\hat y}^\top H,\quad \bar b_2 = \textstyle\sum_n \bar{\hat y}_n,\quad
\bar H = \bar{\hat y} W_2,\quad \bar Z_1 = \bar H \odot (1 - H^2),\quad
\bar W_1 = \bar Z_1^\top X,\quad \bar b_1 = \textstyle\sum_n \bar Z_{1,n}.
$$

In [ ]:
X = rng.normal(size=(8, 3))
y = rng.normal(size=(8, 1))
W1 = Tensor(rng.normal(size=(4, 3)), requires_grad=True)
b1 = Tensor(rng.normal(size=4), requires_grad=True)
W2 = Tensor(rng.normal(size=(1, 4)), requires_grad=True)
b2 = Tensor(rng.normal(size=1), requires_grad=True)


def loss_fn(W1, b1, W2, b2):
    h = F.tanh(F.linear(Tensor(X), W1, b1))
    return F.mse_loss(F.linear(h, W2, b2), Tensor(y))


loss_fn(W1, b1, W2, b2).backward()

# The same gradients, written out with NumPy.
z1 = X @ W1.data.T + b1.data
h = np.tanh(z1)
out = h @ W2.data.T + b2.data
out_bar = 2 * (out - y) / out.size
h_bar = out_bar @ W2.data
z1_bar = h_bar * (1 - h**2)
by_hand = {
    "W1": z1_bar.T @ X,
    "b1": z1_bar.sum(axis=0),
    "W2": out_bar.T @ h,
    "b2": out_bar.sum(axis=0),
}
for name, tensor in zip(by_hand, [W1, b1, W2, b2], strict=True):
    error = np.abs(tensor.grad.data - by_hand[name]).max()
    print(f"{name}: max |autodiff - by hand| = {error:.1e}")

In [ ]:
# Central finite differences, in float64.
gradcheck(loss_fn, [W1, b1, W2, b2])

**Finding.** The three computations agree: autodiff and the hand-written
pass perform the same floating-point operations (here they agree exactly),
and the finite differences agree within their tolerance.

**Exercise.** Replace `tanh` by `F.relu` and update the hand-written pass.
Where can `gradcheck` fail for ReLU, and why?